# 07_巧用排序算法优化MoE融合算子

本章通过一个可运行的 Ascend C 教学实验，理解 MoE 路由中的专家选择、Token 重排和结果还原。实验比较 TopK、QuickSort 和 HeapSort 三种选择策略，并观察它们进入同一条 Permute/Unpermute 融合路径后的差异。

## 1. 实验背景

MoE 层先为每个 token 计算它对多个 expert 的路由分数，再选择少量 expert 执行后续 FFN。选择阶段通常只需要保留 `top_k` 个候选，但后续 token 分派需要按照 expert 对 token 进行分桶和重排。排序算法因此会影响路由开销与融合路径的端到端时间。

## 2. MoE 路由数据结构

输入 logits 的形状为 `[T, E]`，其中 `T` 是 token 数，`E` 是 expert 数。选择结果是 `[T, K]` 的 expert indices 和路由权重；随后将 `(token_id, slot_id)` 按 expert 分组形成 `sortedOrder`，供 Permute kernel 搬运 token。

## 3. 三种排序策略

- **TopK 基线**：维护大小为 K 的最小堆，复杂度约为 `O(E log K)`。
- **QuickSort**：对全部 expert 分数排序，再取前 K 个结果，便于展示完整排序代价。
- **HeapSort**：先构造最大堆，再重复提取前 K 个元素，展示堆排序在选择问题中的应用。

本实验固定 `K=2`，三种策略使用相同的 tie-break 规则：分数相同时选择 expert id 较小者。

## 4. Ascend C 执行模型与 910B 适配

Host 侧根据输入 shape 计算 `tokensPerCore` / `rowsPerCore`，并通过 tiling 设置 `blockDim`。910B 默认使用 `BLOCK_DIM=16`，310B 使用 `BLOCK_DIM=8`；Kernel 只读取 tiling 参数，因此算法代码不依赖具体芯片型号。

## 5. 路由拓扑

<div style="text-align: left;">
  <img src="./images/moe_routing.svg" alt="MoE 排序路由与融合路径" width="720">
</div>

图中三种路由策略是可替换的选择阶段；它们共享后续的 token 搬运路径。

## 6. 学习目标与章节内容

完成本章后，你将能够：

1. 描述 MoE 路由中 expert selection、Permute 和 Unpermute 的数据关系。
2. 解释 TopK、QuickSort、HeapSort 的适用场景和复杂度。
3. 阅读 Ascend C Kernel 与 Host tiling 代码。
4. 使用 910B 目标构建自定义算子，并区分 kernel 时间与端到端时间。

下一步：进入 [07.02 动手实验](./07.02_moe_sort_lab.ipynb)，完成工程检查、构建和数据分析；最后使用 [07.03 章节测试](./07.03_chapter_test.ipynb) 检查理解。